# Solutions — Batch Processing: Batching, Backfills, Hash Partitioning, Compaction, Resumable Jobs

Exercise notebook: `11_python_for_aws/03_batch_processing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
S3_ROOT = LAB / "datasets/s3"        # local stand-in for S3: <root>/<bucket>/<key>
for p in sorted(S3_ROOT.rglob("*"))[:12]:
    print(p.relative_to(S3_ROOT))

## Exercise 1 — 🟢 Easy · Batch by count and size

**Working solution**

In [ ]:
def plan_batches(items, max_items, max_bytes, size=len):
    batches, current, current_bytes = [], [], 0
    for item in items:
        n = size(item)
        if n > max_bytes:
            raise ValueError(f"item of {n} bytes exceeds max_bytes={max_bytes}")
        if current and (len(current) >= max_items or current_bytes + n > max_bytes):
            batches.append(current)
            current, current_bytes = [], 0
        current.append(item)
        current_bytes += n
    if current:
        batches.append(current)
    return batches

**Explanation**

Greedy packing with two constraints; flush *before* adding the item that would break a limit. `size` is pluggable (`len`, `lambda r: len(json.dumps(r))`).

**Complexity:** O(n).

**Common mistakes**

- Forgetting the final partial batch.
- Checking limits after appending.

## Exercise 2 — 🟡 Medium · Plan a backfill

**Working solution**

In [ ]:
from datetime import date, timedelta

def plan_backfill(start, end, done, wave_size):
    first, last = date.fromisoformat(start), date.fromisoformat(end)
    if first > last:
        raise ValueError("start must be <= end")
    missing = [(first + timedelta(days=i)).isoformat() for i in range((last - first).days + 1)]
    missing = [d for d in missing if d not in done]
    return [missing[i:i + wave_size] for i in range(0, len(missing), wave_size)]

**Explanation**

Date arithmetic with `timedelta`, a set for O(1) 'already done' checks, and slicing into waves. Real orchestrators (Airflow `backfill`, Step Functions Map with `MaxConcurrency`) do exactly this.

**Complexity:** O(days).

**Common mistakes**

- Generating date strings by hand and getting month ends wrong.

## Exercise 3 — 🟡 Medium · Deterministic hash partitioning and skew

**Working solution**

In [ ]:
import zlib

def hash_partition(records, key, n):
    parts = [[] for _ in range(n)]
    for r in records:
        parts[zlib.crc32(str(r[key]).encode("utf-8")) % n].append(r)
    return parts

def skew_ratio(partitions):
    sizes = [len(p) for p in partitions]
    total = sum(sizes)
    if not partitions or total == 0:
        return 1.0
    return round(max(sizes) / (total / len(sizes)), 2)

**Explanation**

Same key → same partition is what makes distributed group-bys and joins work (Spark's `HashPartitioner`). Python's `hash('x')` changes between processes (PYTHONHASHSEED), so never use it for
data placement. Low-cardinality keys (7 countries into 8 partitions) guarantee skew — at least one partition is empty.

**Complexity:** O(n).

**Common mistakes**

- `[[]] * n` → n references to the *same* list.
- Using `hash()` for persistent partitioning.

## Exercise 4 — 🟠 Hard · Compact small files

**Working solution**

In [ ]:
import csv
import shutil
import uuid
from pathlib import Path

def compact_partition(partition_dir, target_rows):
    part_dir = Path(partition_dir)
    sources = sorted(part_dir.glob("part-*.csv"))
    header, rows = None, []
    for src in sources:
        with src.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            h = next(reader, None)
            if h is None:
                continue
            if header is None:
                header = h
            elif h != header:
                raise ValueError(f"header mismatch in {src.name}")
            rows.extend(reader)
    staging = part_dir / f".compact-{uuid.uuid4().hex}"
    staging.mkdir()
    try:
        outputs = []
        for i in range(0, len(rows), target_rows):
            out = staging / f"part-c{len(outputs):05d}.csv"
            with out.open("w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(header)
                w.writerows(rows[i:i + target_rows])
            outputs.append(out)
        for src in sources:
            src.unlink()
        for out in outputs:
            out.rename(part_dir / out.name)
    finally:
        shutil.rmtree(staging, ignore_errors=True)
    return {"files_before": len(sources), "files_after": len(outputs), "rows": len(rows)}

**Explanation**

Validation happens before any destructive step; output is staged and swapped. (This loads one partition into memory — fine when partitions are sized sanely; otherwise stream with rolling writers
as in Level 8.) Table formats like Iceberg/Delta make compaction a transactional metadata operation (`OPTIMIZE`).

**Complexity:** O(rows in partition).

**Common mistakes**

- Deleting sources before the compacted files are fully written.
- Losing `_SUCCESS`/other markers.

## Challenge — 🔴 Very Hard · A resumable batch job

**Working solution**

In [ ]:
import hashlib
import json
import os
import tempfile
from pathlib import Path

def _atomic_write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)

def process_resumable(items, batch_size, process_batch, state_path, output_dir):
    state_path, output_dir = Path(state_path), Path(output_dir)
    fingerprint = hashlib.sha256((json.dumps(items, sort_keys=True, default=str) + f"|{batch_size}").encode()).hexdigest()
    state = json.loads(state_path.read_text()) if state_path.exists() else {"fingerprint": fingerprint, "completed": []}
    if state["fingerprint"] != fingerprint:
        raise ValueError("state belongs to a different input; delete it or use another state path")
    completed = set(state["completed"])
    batches = [items[i:i + batch_size] for i in range(0, len(items), batch_size)]
    ran, skipped = [], []
    for index, batch in enumerate(batches):
        if index in completed:
            skipped.append(index)
            continue
        result = process_batch(batch)
        _atomic_write(output_dir / f"batch-{index:05d}.json", json.dumps(result, default=str))
        completed.add(index)
        state["completed"] = sorted(completed)
        _atomic_write(state_path, json.dumps(state, indent=2))
        ran.append(index)
    if len(completed) == len(batches):
        _atomic_write(output_dir / "_SUCCESS", "")
    return {"ran": ran, "skipped": skipped, "total_batches": len(batches)}

**Explanation**

Output first, then state — so a crash between the two at worst redoes one batch, and since each batch's output file is overwritten atomically, redoing it is harmless (idempotent).
The fingerprint prevents a subtle disaster: resuming yesterday's progress against today's different input.

**Complexity:** O(n) plus one state write per batch.

**Common mistakes**

- Saving progress only at the end → a crash at 99% redoes everything.
- Appending outputs → duplicates after resume.